# Esteira de Aprendizado de Máquina — Poker Hand Dataset

**Dataset:** [Poker Hand Dataset — UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/158/poker+hand)

**Objetivo:** Classificar mãos de poker (0 a 9) com base nos naipes e valores de 5 cartas.

**Etapas da esteira:**
1. Carregamento e inspeção dos dados
2. Estatísticas descritivas
3. Limpeza e transformações (colunas e linhas)
4. Divisão em treino, validação e teste
5. Treinamento do modelo
6. Avaliação (matriz de confusão e acurácia)
7. Predição com o modelo implantado

## 1. Importações e Configurações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
import joblib

print('Bibliotecas carregadas com sucesso!')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')

## 2. Carregamento dos Dados

O dataset contém **25.010 exemplos de treino** e **1.000.000 de exemplos de teste**.
Cada linha representa uma mão de poker com 5 cartas (10 atributos: naipe + valor de cada carta).
O alvo (`CLASS`) é a classificação da mão (0 = nada, 1 = um par, ..., 9 = royal flush).

In [ ]:
# Nomes das colunas conforme documentação do dataset
columns = [
    'S1', 'C1',   # Naipe e valor da carta 1
    'S2', 'C2',   # Naipe e valor da carta 2
    'S3', 'C3',   # Naipe e valor da carta 3
    'S4', 'C4',   # Naipe e valor da carta 4
    'S5', 'C5',   # Naipe e valor da carta 5
    'CLASS'       # Classe (mão de poker)
]

# Carregando os arquivos
df_train_raw = pd.read_csv('poker-hand-training-true.data', header=None, names=columns)
df_test_raw  = pd.read_csv('poker-hand-testing.data',       header=None, names=columns)

# Juntamos tudo para ter mais dados e dividirmos nós mesmos
df = pd.concat([df_train_raw, df_test_raw], ignore_index=True)

print(f'Total de exemplos: {len(df):,}')
print(f'Atributos: {df.shape[1]}')
print()
print('Primeiras linhas:')
df.head()

## 3. Estatísticas Descritivas Gerais

In [ ]:
print('=== INFORMAÇÕES GERAIS ===')
print(df.info())

In [ ]:
print('=== ESTATÍSTICAS DESCRITIVAS ===')
df.describe().round(2)

In [ ]:
print('=== VALORES NULOS POR COLUNA ===')
print(df.isnull().sum())
print(f'\nTotal de valores nulos: {df.isnull().sum().sum()}')

In [ ]:
# Dicionário com nomes das classes
class_names = {
    0: 'Nada',
    1: 'Um par',
    2: 'Dois pares',
    3: 'Trinca',
    4: 'Sequência',
    5: 'Flush',
    6: 'Full House',
    7: 'Quadra',
    8: 'Straight Flush',
    9: 'Royal Flush'
}

print('=== DISTRIBUIÇÃO DAS CLASSES ===')
class_dist = df['CLASS'].value_counts().sort_index()
class_df = pd.DataFrame({
    'Classe': [class_names[i] for i in class_dist.index],
    'Quantidade': class_dist.values,
    'Percentual (%)': (class_dist.values / len(df) * 100).round(4)
}, index=class_dist.index)
print(class_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribuição das classes
class_dist.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição das Classes (escala linear)', fontsize=13)
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Quantidade')
axes[0].set_xticklabels([class_names[i] for i in class_dist.index], rotation=35, ha='right')

# Escala log para visualizar classes raras
class_dist.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white', logy=True)
axes[1].set_title('Distribuição das Classes (escala log)', fontsize=13)
axes[1].set_xlabel('Classe')
axes[1].set_ylabel('Quantidade (log)')
axes[1].set_xticklabels([class_names[i] for i in class_dist.index], rotation=35, ha='right')

plt.tight_layout()
plt.show()
print('Nota: o dataset é fortemente desbalanceado — "Nada" e "Um par" dominam.')

In [ ]:
# Correlação entre atributos
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr().round(2), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Matriz de Correlação — Poker Hand Dataset', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Limpeza e Transformações

### 4.1 Transformação em COLUNAS — Engenharia de Atributos

Os naipes (S1–S5) são variáveis nominais mas estão codificadas como inteiros (1–4), o que pode
induzir o modelo a assumir uma ordem inexistente. Vamos criar **features derivadas** mais
expressivas para ajudar o classificador a aprender padrões de poker:

- `num_suits_distintos`: quantos naipes diferentes aparecem na mão
- `flush_potencial`: 1 se todos os 5 naipes são iguais (flush)
- `max_valor` e `min_valor`: maior e menor valor entre as 5 cartas
- `range_valores`: diferença entre maior e menor valor
- `valores_distintos`: quantos valores de carta são diferentes
- `rank_std`: desvio padrão dos valores (mede diversidade da mão)

In [ ]:
def engenharia_atributos(df):
    df = df.copy()

    suits  = df[['S1','S2','S3','S4','S5']]
    ranks  = df[['C1','C2','C3','C4','C5']]

    # --- Transformações em colunas (novas features) ---
    df['num_suits_distintos'] = suits.nunique(axis=1)
    df['flush_potencial']     = (df['num_suits_distintos'] == 1).astype(int)
    df['max_valor']           = ranks.max(axis=1)
    df['min_valor']           = ranks.min(axis=1)
    df['range_valores']       = df['max_valor'] - df['min_valor']
    df['valores_distintos']   = ranks.nunique(axis=1)
    df['rank_std']            = ranks.std(axis=1).round(4)

    return df

df = engenharia_atributos(df)

print('Novas colunas criadas:')
novas = ['num_suits_distintos','flush_potencial','max_valor','min_valor',
         'range_valores','valores_distintos','rank_std']
print(df[novas].describe().round(2))

### 4.2 Transformação em LINHAS — Remoção de duplicatas

Com 1 milhão de linhas é comum haver mãos de poker repetidas. Vamos remover duplicatas
para garantir exemplos únicos e evitar vazamento de dados entre treino e teste.

In [ ]:
print(f'Linhas antes da remoção de duplicatas: {len(df):,}')

# Verificamos duplicatas com base somente nos atributos originais das cartas
cols_cartas = ['S1','C1','S2','C2','S3','C3','S4','C4','S5','C5']
df = df.drop_duplicates(subset=cols_cartas).reset_index(drop=True)

print(f'Linhas após remoção de duplicatas:    {len(df):,}')
print(f'Duplicatas removidas: {1025010 - len(df):,}')

In [ ]:
print('Dataset final:')
print(f'  Linhas:   {df.shape[0]:,}')
print(f'  Colunas:  {df.shape[1]}')
df.head(3)

## 5. Divisão em Treino, Validação e Teste

Estratégia: **Hold-out** com estratificação pela classe (para preservar proporção das classes raras).

- **Treino:** 70%
- **Validação:** 15%
- **Teste:** 15%

In [ ]:
# Features e alvo
feature_cols = [
    'S1','C1','S2','C2','S3','C3','S4','C4','S5','C5',
    'num_suits_distintos','flush_potencial','max_valor','min_valor',
    'range_valores','valores_distintos','rank_std'
]
X = df[feature_cols]
y = df['CLASS']

# 1ª divisão: separa teste (15%)
X_temp, X_teste, y_temp, y_teste = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# 2ª divisão: separa validação (15% do total = ~17.6% do restante)
X_treino, X_val, y_treino, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp
)

print('Divisão dos dados:')
print(f'  Treino:    {len(X_treino):>8,} exemplos ({len(X_treino)/len(X)*100:.1f}%)')
print(f'  Validação: {len(X_val):>8,} exemplos ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Teste:     {len(X_teste):>8,} exemplos ({len(X_teste)/len(X)*100:.1f}%)')

## 6. Treinamento do Modelo

Utilizamos **Random Forest**, um ensemble de árvores de decisão robusto para datasets desbalanceados.
O parâmetro `class_weight='balanced'` ajuda o modelo a não ignorar classes raras como Royal Flush.

In [ ]:
print('Treinando Random Forest...')

modelo = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1          # usa todos os núcleos disponíveis
)

modelo.fit(X_treino, y_treino)
print('Modelo treinado!')

# Avaliação no conjunto de validação
y_pred_val = modelo.predict(X_val)
acc_val = accuracy_score(y_val, y_pred_val)
print(f'\nAcurácia no conjunto de VALIDAÇÃO: {acc_val:.4f} ({acc_val*100:.2f}%)')

In [ ]:
# Importância dos atributos
importancias = pd.Series(modelo.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importancias.plot(kind='bar', color='mediumseagreen', edgecolor='white')
plt.title('Importância dos Atributos — Random Forest', fontsize=13)
plt.ylabel('Importância')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

## 7. Avaliação no Conjunto de Teste

Agora avaliamos o modelo no conjunto de **teste** (dados que o modelo nunca viu).

In [ ]:
y_pred_teste = modelo.predict(X_teste)
acc_teste = accuracy_score(y_teste, y_pred_teste)

print(f'Acurácia no conjunto de TESTE: {acc_teste:.4f} ({acc_teste*100:.2f}%)')
print()
print('=== RELATÓRIO DE CLASSIFICAÇÃO ===')
target_names = [class_names[i] for i in sorted(y.unique())]
print(classification_report(y_teste, y_pred_teste, target_names=target_names))

### 7.1 Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_teste, y_pred_teste)

fig, ax = plt.subplots(figsize=(12, 9))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[class_names[i] for i in sorted(y.unique())]
)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
plt.title('Matriz de Confusão — Conjunto de Teste', fontsize=14)
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

print()
print('Interpretação: linhas = classe real | colunas = classe prevista')
print('A diagonal principal mostra os acertos; fora dela são os erros.')

In [ ]:
# Matriz de confusão normalizada (facilita comparação entre classes desbalanceadas)
cm_norm = confusion_matrix(y_teste, y_pred_teste, normalize='true')

fig, ax = plt.subplots(figsize=(12, 9))
disp_norm = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=[class_names[i] for i in sorted(y.unique())]
)
disp_norm.plot(ax=ax, cmap='Greens', colorbar=True)
plt.title('Matriz de Confusão Normalizada (por linha) — Conjunto de Teste', fontsize=13)
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

## 8. Implantação e Predição com o Modelo

Salvamos o modelo treinado em disco (simulando implantação) e o carregamos para fazer predições
em novas mãos de poker, como faria um sistema em produção.

In [ ]:
# Salva o modelo
joblib.dump(modelo, 'modelo_poker_hand.pkl')
print('Modelo salvo em: modelo_poker_hand.pkl')

# Carrega o modelo (simulando ambiente de produção)
modelo_carregado = joblib.load('modelo_poker_hand.pkl')
print('Modelo carregado com sucesso!')

In [ ]:
def preparar_mao(cartas_raw):
    """
    Recebe uma lista com os valores brutos da mão:
    [S1, C1, S2, C2, S3, C3, S4, C4, S5, C5]
    Retorna DataFrame com todas as features (originais + engenharia).
    """
    cols = ['S1','C1','S2','C2','S3','C3','S4','C4','S5','C5']
    df_mao = pd.DataFrame([cartas_raw], columns=cols)
    df_mao = engenharia_atributos(df_mao)
    df_mao.drop(columns=['CLASS'], errors='ignore', inplace=True)
    return df_mao[feature_cols]


def prever_mao(cartas_raw):
    """
    Faz a predição e exibe resultado legível.
    cartas_raw: [S1, C1, S2, C2, S3, C3, S4, C4, S5, C5]
      S = naipe (1=Copas, 2=Espadas, 3=Ouros, 4=Paus)
      C = valor  (1=Ás, 2-10=número, 11=Valete, 12=Dama, 13=Rei)
    """
    X_novo = preparar_mao(cartas_raw)
    classe_pred  = modelo_carregado.predict(X_novo)[0]
    probabilidades = modelo_carregado.predict_proba(X_novo)[0]

    suit_map = {1:'♥ Copas', 2:'♠ Espadas', 3:'♦ Ouros', 4:'♣ Paus'}
    rank_map = {1:'Ás',2:'2',3:'3',4:'4',5:'5',6:'6',7:'7',
                8:'8',9:'9',10:'10',11:'Valete',12:'Dama',13:'Rei'}

    print('=== PREDIÇÃO ===')
    print('Mão de poker:')
    for i in range(5):
        s = cartas_raw[i*2]
        c = cartas_raw[i*2+1]
        print(f'  Carta {i+1}: {rank_map[c]} de {suit_map[s]}')

    print(f'\nClassificação prevista: [{classe_pred}] {class_names[classe_pred].upper()}')
    print(f'Confiança: {probabilidades[classe_pred]*100:.1f}%')
    print()
    print('Probabilidades por classe:')
    for cls, prob in sorted(enumerate(probabilidades), key=lambda x: -x[1]):
        bar = '█' * int(prob * 30)
        print(f'  [{cls}] {class_names[cls]:<16} {prob*100:5.1f}% {bar}')

print('Funções de predição definidas!')

In [ ]:
# --- PREDIÇÃO 1: Royal Flush (a melhor mão possível) ---
# Ás, Rei, Dama, Valete e 10 — todos de Copas
royal_flush = [1,1, 1,13, 1,12, 1,11, 1,10]
prever_mao(royal_flush)

In [ ]:
# --- PREDIÇÃO 2: Um par ---
# Dois Ases + cartas aleatórias de naipes diferentes
um_par = [1,1, 2,1, 3,7, 4,9, 2,13]
prever_mao(um_par)

In [ ]:
# --- PREDIÇÃO 3: Mão fraca (nada) ---
# Cartas de naipes e valores variados sem combinação
nada = [1,2, 2,5, 3,9, 4,11, 1,13]
prever_mao(nada)

In [ ]:
# --- PREDIÇÃO 4: Flush (5 cartas do mesmo naipe) ---
flush = [3,2, 3,7, 3,9, 3,11, 3,4]
prever_mao(flush)

## 9. Resumo Final

In [ ]:
print('=' * 55)
print('         RESUMO DA ESTEIRA DE ML — POKER HAND')
print('=' * 55)
print(f'Dataset: UCI Poker Hand Dataset')
print(f'Total de exemplos (após dedup): {len(df):,}')
print(f'Features utilizadas:            {len(feature_cols)}')
print(f'  - Originais:                  10')
print(f'  - Engenharia de atributos:     7')
print()
print(f'Divisão (Hold-out estratificado):')
print(f'  Treino:     {len(X_treino):,} exemplos (70%)')
print(f'  Validação:  {len(X_val):,} exemplos (15%)')
print(f'  Teste:      {len(X_teste):,} exemplos (15%)')
print()
print(f'Modelo: Random Forest (200 estimadores, class_weight=balanced)')
print(f'Acurácia na Validação: {acc_val*100:.2f}%')
print(f'Acurácia no Teste:     {acc_teste*100:.2f}%')
print('=' * 55)